# Laboratorio 04 â€” Vistas y OptimizaciÃ³n SQL sobre tu propio dataset

**Semana:** 03 | **Actividad de referencia:** Actividad 04  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

Aplica los conceptos de `CREATE OR REPLACE VIEW`, `EXPLAIN` y equivalencias SQL â†” PySpark de la Actividad 04 sobre tu dataset personal. El objetivo es aprender a organizar consultas reutilizables y a leer el plan de ejecuciÃ³n para identificar cuellos de botella.

## Parte 1 â€” DescripciÃ³n del dataset

1. **Nombre, fuente y URL** del dataset.
2. **Vista que crearÃ¡s:** Â¿QuÃ© consulta de negocio encapsularÃ¡ la vista? Â¿Por quÃ© tiene sentido reutilizarla?
3. **Preguntas de negocio** que responderÃ¡s consultando la vista (no la tabla base directamente).

**Escribe tu respuesta aquÃ­:**

## Parte 2 â€” Cargar el dataset como tabla Delta

In [ ]:
VOL     = "/Volumes/workspace/default/week_3"  # ajusta si usas otra ubicaciÃ³n
ARCHIVO = "tu_archivo.csv"
TABLA   = "workspace.default.lab03_04_mi_dataset"

df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(f"{VOL}/{ARCHIVO}")

df.write.format("delta").mode("overwrite").saveAsTable(TABLA)
print(f"âœ“ {TABLA}: {df.count():,} filas x {len(df.columns)} columnas")

## Parte 3 â€” Perfil tÃ©cnico

In [ ]:
spark.sql(f"DESCRIBE TABLE EXTENDED {TABLA}").show(30, truncate=False)

In [ ]:
# Historial Delta de la tabla
spark.sql(f"DESCRIBE HISTORY {TABLA}").show(5, truncate=False)

## Parte 4 â€” Crear Vistas SQL

Crea al menos 2 vistas que encapsulen lÃ³gica reutilizable de tu dataset.

In [ ]:
# Vista 1: resumen de negocio base
# Define quÃ© lÃ³gica encapsula esta vista y por quÃ© serÃ­a Ãºtil reutilizarla
spark.sql(f"""
    CREATE OR REPLACE VIEW workspace.default.v_lab03_resumen AS
    SELECT
        columna_categoria,
        COUNT(*)              AS total_registros,
        AVG(columna_numerica) AS promedio,
        SUM(columna_numerica) AS total
    FROM {TABLA}
    WHERE columna_clave IS NOT NULL
    GROUP BY columna_categoria
""")
print("âœ“ Vista v_lab03_resumen creada")

In [ ]:
# Consultar la vista
spark.sql("SELECT * FROM workspace.default.v_lab03_resumen ORDER BY total DESC LIMIT 15").show(truncate=False)

In [ ]:
# Vista 2: vista analÃ­tica mÃ¡s compleja (usa window function o JOIN si aplica)
spark.sql(f"""
    CREATE OR REPLACE VIEW workspace.default.v_lab03_analitica AS
    SELECT
        *,
        RANK() OVER (
            PARTITION BY columna_categoria
            ORDER BY columna_numerica DESC
        ) AS ranking
    FROM {TABLA}
""")
print("âœ“ Vista v_lab03_analitica creada")

In [ ]:
# Consultar la vista analÃ­tica â€” solo top 1 por categorÃ­a
spark.sql("""
    SELECT *
    FROM workspace.default.v_lab03_analitica
    WHERE ranking = 1
    ORDER BY columna_categoria
""").show(truncate=False)

**Observaciones sobre las vistas:** Â¿Las vistas almacenan datos o solo la definiciÃ³n de la consulta? Â¿QuÃ© pasa si modificas la tabla base despuÃ©s de crear la vista?

## Parte 5 â€” EXPLAIN: Analizar el plan de ejecuciÃ³n

In [ ]:
# Plan de ejecuciÃ³n de la consulta directa a la tabla
spark.sql(f"""
    EXPLAIN FORMATTED
    SELECT columna_categoria, COUNT(*), AVG(columna_numerica)
    FROM {TABLA}
    GROUP BY columna_categoria
    ORDER BY 2 DESC
""").show(100, truncate=False)

In [ ]:
# Plan de ejecuciÃ³n de consulta sobre la vista
spark.sql("""
    EXPLAIN FORMATTED
    SELECT * FROM workspace.default.v_lab03_resumen WHERE total > 10
""").show(100, truncate=False)

**AnÃ¡lisis del plan:**
1. Â¿Ves algÃºn `FileScan` en el plan? Â¿CuÃ¡ntos archivos escanea?
2. Â¿Hay algÃºn `Exchange` (shuffle)? Â¿En quÃ© parte del plan aparece y por quÃ©?
3. Â¿El plan de la consulta sobre la vista es igual al de la consulta directa? Â¿Por quÃ©?  
4. Â¿QuÃ© optimizaciÃ³n de Spark Catalyst puedes identificar en el plan?

## Parte 6 â€” Equivalencias SQL â†” PySpark

Para cada operaciÃ³n SQL escribe su equivalente exacto en PySpark y viceversa.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Equivalente PySpark de: SELECT columna_categoria, COUNT(*), AVG(columna_numerica)
#                          FROM tabla GROUP BY columna_categoria ORDER BY 2 DESC
df = spark.table(TABLA)

df_equiv = df.groupBy("columna_categoria") \
    .agg(
        F.count("*").alias("total_registros"),
        F.avg("columna_numerica").alias("promedio")
    ) \
    .orderBy(F.col("total_registros").desc())

df_equiv.show(15, truncate=False)

In [ ]:
# Equivalente PySpark de la window function SQL
# SQL: RANK() OVER (PARTITION BY columna_categoria ORDER BY columna_numerica DESC)
w = Window.partitionBy("columna_categoria").orderBy(F.col("columna_numerica").desc())

df.withColumn("ranking", F.rank().over(w)) \
    .filter(F.col("ranking") == 1) \
    .orderBy("columna_categoria") \
    .show(truncate=False)

**Tabla comparativa SQL â†” PySpark:**

| Concepto SQL | Equivalente PySpark |
|---|---|
| `WHERE col IS NOT NULL` | `.filter(F.col('col').isNotNull())` |
| `GROUP BY col` | `.groupBy('col')` |
| `HAVING COUNT(*) > N` | `.filter(F.col('count') > N)` |
| `ORDER BY col DESC` | `.orderBy(F.col('col').desc())` |
| `CREATE OR REPLACE VIEW v AS ...` | `df.createOrReplaceTempView('v')` |
| `RANK() OVER (PARTITION BY ...)` | `F.rank().over(Window.partitionBy(...))` |
| AÃ±ade mÃ¡s filas | para tu dataset |

## Parte 7 â€” Preguntas de negocio consultando las vistas

In [ ]:
# Pregunta 1 â€” usando v_lab03_resumen
spark.sql("""
    SELECT *
    FROM workspace.default.v_lab03_resumen
    -- aÃ±ade tus condiciones
    LIMIT 10
""").show(truncate=False)

**ConclusiÃ³n pregunta 1:**

In [ ]:
# Pregunta 2 â€” usando v_lab03_analitica
spark.sql("""
    SELECT *
    FROM workspace.default.v_lab03_analitica
    -- aÃ±ade tus condiciones
    LIMIT 10
""").show(truncate=False)

**ConclusiÃ³n pregunta 2:**

## Parte 8 â€” ReflexiÃ³n final

1. Â¿CuÃ¡ndo usarÃ­as una vista en lugar de una tabla materializada (CTAS)?
2. Â¿QuÃ© encontraste en el plan `EXPLAIN` que no esperabas?
3. Â¿CuÃ¡ndo preferirÃ­as SQL puro sobre PySpark para escribir transformaciones en producciÃ³n?
4. Si tuvieras que optimizar tu consulta mÃ¡s lenta, Â¿quÃ© harÃ­as basÃ¡ndote en el plan `EXPLAIN`?

---

## Entrega en Git

```bash
# Copia el template a tu carpeta (solo la primera vez)
# cp semana_03/laboratorios/lab_04_vistas_optimizacion.ipynb semana_03/laboratorios/<tu-nombre>/lab_04_vistas_optimizacion.ipynb

git add semana_03/laboratorios/<tu-nombre>/lab_04_vistas_optimizacion.ipynb
git commit -m "lab: semana03 lab04 vistas EXPLAIN SQL-PySpark <nombre-dataset> - <tu-nombre>"
git push origin develop
```